
# Medical LLM Notebook 6B — Length-Normalized Structural Validation

## Purpose
Notebook 6 showed that several graph features are strongly associated with response length.  
This notebook therefore performs a **length-normalized paired analysis** before any hypergraph or fuzzy-risk modeling.

### Primary idea
For each structural feature \(X\), estimate the expected value from response length only:

\[
\widehat{X}=f(\log(1+n)),
\]

where \(n\) is the number of response words.

The normalized score is the cross-fitted residual:

\[
X^{*}=X-\widehat{X}.
\]

Five-fold **GroupKFold by Pair_ID** is used so the factual and hallucinated responses from the same medical question always remain in the same fold. The normalization model never uses the factual/hallucinated label.

The primary normalization uses a cubic spline of response length. A simpler linear model is also used as a sensitivity analysis.

### Primary inferential test
For every feature:

\[
\Delta X^{*}
=
X^{*}_{Hallucinated}
-
X^{*}_{Factual}.
\]

The paired normalized differences are tested with the Wilcoxon signed-rank test, followed by Holm correction across features.

**Important:** `Words` is treated only as a nuisance/control variable, not as a hallucination predictor.


In [ ]:

!pip -q install -U scipy scikit-learn statsmodels

import os
import glob
import zipfile
import warnings

import numpy as np
import pandas as pd

from google.colab import files
from scipy.stats import wilcoxon, rankdata, spearmanr
from statsmodels.stats.multitest import multipletests

from sklearn.model_selection import GroupKFold
from sklearn.preprocessing import SplineTransformer
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LinearRegression

warnings.filterwarnings("ignore")

SEED = 42
N_SPLITS = 5

print("Notebook 6B libraries loaded.")


## 1. Upload Notebook 5 outputs

In [ ]:

uploaded = files.upload()

zip_files = [
    name for name in uploaded.keys()
    if name.lower().endswith(".zip")
]

assert len(zip_files) >= 1, (
    "Please upload Notebook5_100Pair_Outputs.zip"
)

ZIP_FILE = zip_files[0]

EXTRACT_DIR = "/content/Notebook6B_input"
os.makedirs(EXTRACT_DIR, exist_ok=True)

with zipfile.ZipFile(ZIP_FILE, "r") as z:
    z.extractall(EXTRACT_DIR)

print("Extracted:", ZIP_FILE)


## 2. Load and validate the 100-pair data

In [ ]:

response_file = glob.glob(
    "/content/Notebook6B_input/**/MedHallu_200_Response_Features.csv",
    recursive=True
)[0]

paired_file = glob.glob(
    "/content/Notebook6B_input/**/MedHallu_100_Paired_Differences.csv",
    recursive=True
)[0]

responses = pd.read_csv(response_file)
paired_raw = pd.read_csv(paired_file)

print("Response-level shape:", responses.shape)
print("Paired shape:", paired_raw.shape)

assert len(responses) == 200
assert responses["Pair_ID"].nunique() == 100
assert set(responses["Response_Type"]) == {
    "Factual",
    "Hallucinated"
}

pair_counts = (
    responses
    .groupby("Pair_ID")["Response_Type"]
    .nunique()
)

assert (pair_counts == 2).all()
assert responses.isna().sum().sum() == 0

print("Validation passed: 100 complete factual–hallucinated pairs.")


## 3. Define structural features

In [ ]:

FEATURES = [
    "GCC",
    "Communication_Efficiency",
    "Mean_Edge_Weight",
    "Max_PageRank",
    "PageRank_Concentration",
    "Mean_Betweenness",
    "Max_Betweenness",
    "Max_GCCDC",
    "Mean_Structural_Influence",
    "Max_Structural_Influence",
    "QAS_Mean",
    "QAS_Median",
    "QAS_Std"
]

CONTROL = "Words"

missing = [
    f for f in FEATURES + [CONTROL]
    if f not in responses.columns
]

assert len(missing) == 0, f"Missing columns: {missing}"

print("Primary structural features:", len(FEATURES))
print("Length control:", CONTROL)



## 4. Helper functions

The rank-biserial correlation is reported as an effect size for the paired Wilcoxon test. Positive values indicate that the normalized hallucinated score tends to exceed the normalized factual score.


In [ ]:

def signed_rank_biserial(delta):

    delta = np.asarray(delta, dtype=float)
    delta = delta[np.isfinite(delta)]
    delta = delta[delta != 0]

    if len(delta) == 0:
        return np.nan

    ranks = rankdata(np.abs(delta))

    w_pos = ranks[delta > 0].sum()
    w_neg = ranks[delta < 0].sum()

    return (w_pos - w_neg) / (w_pos + w_neg)


def bootstrap_ci(
    values,
    statistic=np.mean,
    B=10000,
    seed=42
):

    values = np.asarray(values, dtype=float)
    values = values[np.isfinite(values)]

    rng = np.random.default_rng(seed)

    boot = np.empty(B)

    for b in range(B):

        sample = rng.choice(
            values,
            size=len(values),
            replace=True
        )

        boot[b] = statistic(sample)

    return np.quantile(
        boot,
        [0.025, 0.975]
    )



## 5. Cross-fitted cubic-spline normalization

A flexible length-only model is fitted:

\[
X=f(\log(1+n))+\varepsilon.
\]

The factual/hallucinated label is **not** included.  
Pair-level grouping prevents information from the same question appearing in both the training and validation fold.


In [ ]:

def crossfit_length_residuals(
    df,
    features,
    model_type="spline",
    n_splits=5
):

    work = df.copy().reset_index(drop=True)

    x = np.log1p(
        work["Words"].astype(float).to_numpy()
    ).reshape(-1, 1)

    groups = work[
        "Pair_ID"
    ].astype(str).to_numpy()

    gkf = GroupKFold(
        n_splits=n_splits
    )

    output = work[
        [
            "Pair_ID",
            "Response_Type",
            "Label",
            "Difficulty",
            "Hallucination_Category",
            "Words"
        ]
    ].copy()

    fold_id = np.full(
        len(work),
        -1,
        dtype=int
    )

    for feature in features:

        y = (
            work[feature]
            .astype(float)
            .to_numpy()
        )

        predictions = np.full(
            len(work),
            np.nan,
            dtype=float
        )

        residual_z = np.full(
            len(work),
            np.nan,
            dtype=float
        )

        for fold, (train_idx, test_idx) in enumerate(
            gkf.split(
                x,
                y,
                groups=groups
            )
        ):

            if model_type == "spline":

                model = make_pipeline(
                    SplineTransformer(
                        n_knots=4,
                        degree=3,
                        include_bias=False
                    ),
                    LinearRegression()
                )

            elif model_type == "linear":

                model = LinearRegression()

            else:
                raise ValueError(
                    "model_type must be 'spline' or 'linear'"
                )

            model.fit(
                x[train_idx],
                y[train_idx]
            )

            pred_train = model.predict(
                x[train_idx]
            )

            pred_test = model.predict(
                x[test_idx]
            )

            train_residual = (
                y[train_idx]
                -
                pred_train
            )

            train_scale = np.std(
                train_residual,
                ddof=1
            )

            if (
                not np.isfinite(train_scale)
                or train_scale <= 1e-12
            ):
                train_scale = 1.0

            predictions[
                test_idx
            ] = pred_test

            residual_z[
                test_idx
            ] = (
                y[test_idx]
                -
                pred_test
            ) / train_scale

            fold_id[
                test_idx
            ] = fold

        output[
            f"{feature}_Predicted_From_Length"
        ] = predictions

        output[
            f"{feature}_LengthNorm"
        ] = residual_z

    output["Fold"] = fold_id

    return output


norm_spline = crossfit_length_residuals(
    responses,
    FEATURES,
    model_type="spline",
    n_splits=N_SPLITS
)

assert not norm_spline.filter(
    like="_LengthNorm"
).isna().any().any()

print(
    "Cross-fitted spline normalization complete."
)

display(
    norm_spline.head()
)


## 6. Check whether normalization removed length dependence

In [ ]:

diagnostic_rows = []

for feature in FEATURES:

    raw_rho, raw_p = spearmanr(
        responses["Words"],
        responses[feature]
    )

    norm_rho, norm_p = spearmanr(
        norm_spline["Words"],
        norm_spline[
            f"{feature}_LengthNorm"
        ]
    )

    diagnostic_rows.append({
        "Feature": feature,
        "Raw_rho_with_Words": raw_rho,
        "Raw_p": raw_p,
        "Normalized_rho_with_Words": norm_rho,
        "Normalized_p": norm_p,
        "Abs_Rho_Reduction":
            abs(raw_rho)
            - abs(norm_rho)
    })

length_diagnostics = pd.DataFrame(
    diagnostic_rows
)

display(
    length_diagnostics
    .sort_values(
        "Abs_Rho_Reduction",
        ascending=False
    )
    .round(6)
)


## 7. Construct paired normalized scores

In [ ]:

f_norm = (
    norm_spline[
        norm_spline[
            "Response_Type"
        ] == "Factual"
    ]
    .set_index("Pair_ID")
)

h_norm = (
    norm_spline[
        norm_spline[
            "Response_Type"
        ] == "Hallucinated"
    ]
    .set_index("Pair_ID")
)

common_pairs = sorted(
    set(f_norm.index)
    &
    set(h_norm.index)
)

assert len(common_pairs) == 100

paired_norm_rows = []

for pair_id in common_pairs:

    record = {
        "Pair_ID": pair_id,
        "Difficulty":
            f_norm.loc[
                pair_id,
                "Difficulty"
            ],
        "Hallucination_Category":
            f_norm.loc[
                pair_id,
                "Hallucination_Category"
            ],
        "Words_Factual":
            float(
                f_norm.loc[
                    pair_id,
                    "Words"
                ]
            ),
        "Words_Hallucinated":
            float(
                h_norm.loc[
                    pair_id,
                    "Words"
                ]
            )
    }

    record["Delta_Words"] = (
        record["Words_Hallucinated"]
        -
        record["Words_Factual"]
    )

    for feature in FEATURES:

        col = (
            f"{feature}_LengthNorm"
        )

        fv = float(
            f_norm.loc[
                pair_id,
                col
            ]
        )

        hv = float(
            h_norm.loc[
                pair_id,
                col
            ]
        )

        record[
            f"{feature}_Factual_Norm"
        ] = fv

        record[
            f"{feature}_Hallucinated_Norm"
        ] = hv

        record[
            f"DeltaNorm_{feature}"
        ] = hv - fv

    paired_norm_rows.append(
        record
    )

paired_norm = pd.DataFrame(
    paired_norm_rows
)

print(
    "Normalized paired dataset:",
    paired_norm.shape
)

display(
    paired_norm.head()
)


## 8. Primary normalized paired tests

In [ ]:

test_rows = []

for feature in FEATURES:

    f = paired_norm[
        f"{feature}_Factual_Norm"
    ].astype(float).to_numpy()

    h = paired_norm[
        f"{feature}_Hallucinated_Norm"
    ].astype(float).to_numpy()

    delta = h - f

    result = wilcoxon(
        h,
        f,
        alternative="two-sided",
        zero_method="wilcox",
        method="auto"
    )

    mean_ci = bootstrap_ci(
        delta,
        statistic=np.mean,
        B=10000,
        seed=SEED
    )

    median_ci = bootstrap_ci(
        delta,
        statistic=np.median,
        B=10000,
        seed=SEED
    )

    sd_delta = np.std(
        delta,
        ddof=1
    )

    test_rows.append({
        "Feature": feature,
        "Factual_Norm_Mean":
            np.mean(f),
        "Hallucinated_Norm_Mean":
            np.mean(h),
        "Mean_Normalized_Delta":
            np.mean(delta),
        "Mean_Delta_CI_Low":
            mean_ci[0],
        "Mean_Delta_CI_High":
            mean_ci[1],
        "Median_Normalized_Delta":
            np.median(delta),
        "Median_Delta_CI_Low":
            median_ci[0],
        "Median_Delta_CI_High":
            median_ci[1],
        "Wilcoxon_W":
            result.statistic,
        "Wilcoxon_p":
            result.pvalue,
        "Rank_Biserial":
            signed_rank_biserial(delta),
        "Cohen_dz":
            np.mean(delta) / sd_delta
            if sd_delta > 0
            else np.nan,
        "Positive_Delta_n":
            int((delta > 0).sum()),
        "Negative_Delta_n":
            int((delta < 0).sum()),
        "Zero_Delta_n":
            int((delta == 0).sum())
    })

normalized_stats = pd.DataFrame(
    test_rows
)

normalized_stats[
    "Wilcoxon_p_Holm"
] = multipletests(
    normalized_stats[
        "Wilcoxon_p"
    ],
    method="holm"
)[1]

normalized_stats[
    "Significant_Holm_0.05"
] = (
    normalized_stats[
        "Wilcoxon_p_Holm"
    ] < 0.05
)

display(
    normalized_stats
    .sort_values(
        "Wilcoxon_p_Holm"
    )
    .round(6)
)


## 9. Are normalized paired differences still associated with length difference?

In [ ]:

delta_length_rows = []

for feature in FEATURES:

    rho, p = spearmanr(
        paired_norm[
            "Delta_Words"
        ],
        paired_norm[
            f"DeltaNorm_{feature}"
        ]
    )

    delta_length_rows.append({
        "Feature": feature,
        "rho_DeltaNorm_vs_DeltaWords": rho,
        "p": p
    })

delta_length_diagnostics = pd.DataFrame(
    delta_length_rows
)

delta_length_diagnostics[
    "p_Holm"
] = multipletests(
    delta_length_diagnostics[
        "p"
    ],
    method="holm"
)[1]

display(
    delta_length_diagnostics
    .sort_values(
        "rho_DeltaNorm_vs_DeltaWords",
        key=lambda s: s.abs(),
        ascending=False
    )
    .round(6)
)



## 10. Linear-normalization sensitivity analysis

The same cross-fitting procedure is repeated using only a linear function of \(\log(1+n)\).  
A result is stronger when its direction and inference are not dependent on the spline specification.


In [ ]:

norm_linear = crossfit_length_residuals(
    responses,
    FEATURES,
    model_type="linear",
    n_splits=N_SPLITS
)

f_lin = (
    norm_linear[
        norm_linear[
            "Response_Type"
        ] == "Factual"
    ]
    .set_index("Pair_ID")
)

h_lin = (
    norm_linear[
        norm_linear[
            "Response_Type"
        ] == "Hallucinated"
    ]
    .set_index("Pair_ID")
)

linear_rows = []

for feature in FEATURES:

    col = (
        f"{feature}_LengthNorm"
    )

    f = (
        f_lin.loc[
            common_pairs,
            col
        ]
        .astype(float)
        .to_numpy()
    )

    h = (
        h_lin.loc[
            common_pairs,
            col
        ]
        .astype(float)
        .to_numpy()
    )

    delta = h - f

    result = wilcoxon(
        h,
        f,
        alternative="two-sided",
        zero_method="wilcox",
        method="auto"
    )

    linear_rows.append({
        "Feature": feature,
        "Mean_Normalized_Delta_Linear":
            np.mean(delta),
        "Median_Normalized_Delta_Linear":
            np.median(delta),
        "Wilcoxon_p_Linear":
            result.pvalue,
        "Rank_Biserial_Linear":
            signed_rank_biserial(delta)
    })

linear_sensitivity = pd.DataFrame(
    linear_rows
)

linear_sensitivity[
    "Wilcoxon_p_Linear_Holm"
] = multipletests(
    linear_sensitivity[
        "Wilcoxon_p_Linear"
    ],
    method="holm"
)[1]

linear_sensitivity[
    "Significant_Linear_Holm_0.05"
] = (
    linear_sensitivity[
        "Wilcoxon_p_Linear_Holm"
    ] < 0.05
)

display(
    linear_sensitivity
    .sort_values(
        "Wilcoxon_p_Linear_Holm"
    )
    .round(6)
)


## 11. Compare spline and linear normalization

In [ ]:

comparison = (
    normalized_stats[
        [
            "Feature",
            "Mean_Normalized_Delta",
            "Wilcoxon_p_Holm",
            "Rank_Biserial",
            "Significant_Holm_0.05"
        ]
    ]
    .merge(
        linear_sensitivity[
            [
                "Feature",
                "Mean_Normalized_Delta_Linear",
                "Wilcoxon_p_Linear_Holm",
                "Rank_Biserial_Linear",
                "Significant_Linear_Holm_0.05"
            ]
        ],
        on="Feature"
    )
    .merge(
        length_diagnostics[
            [
                "Feature",
                "Raw_rho_with_Words",
                "Normalized_rho_with_Words"
            ]
        ],
        on="Feature"
    )
    .merge(
        delta_length_diagnostics[
            [
                "Feature",
                "rho_DeltaNorm_vs_DeltaWords",
                "p_Holm"
            ]
        ],
        on="Feature"
    )
)

comparison[
    "Same_Direction"
] = (
    np.sign(
        comparison[
            "Mean_Normalized_Delta"
        ]
    )
    ==
    np.sign(
        comparison[
            "Mean_Normalized_Delta_Linear"
        ]
    )
)

comparison[
    "Stable_Across_Normalizations"
] = (
    comparison[
        "Significant_Holm_0.05"
    ]
    &
    comparison[
        "Significant_Linear_Holm_0.05"
    ]
    &
    comparison[
        "Same_Direction"
    ]
)

display(
    comparison
    .sort_values(
        [
            "Stable_Across_Normalizations",
            "Wilcoxon_p_Holm"
        ],
        ascending=[
            False,
            True
        ]
    )
    .round(6)
)


## 12. Difficulty-level descriptive analysis after normalization

In [ ]:

difficulty_rows = []

for difficulty, group in paired_norm.groupby(
    "Difficulty"
):

    for feature in FEATURES:

        delta = group[
            f"DeltaNorm_{feature}"
        ].astype(float)

        difficulty_rows.append({
            "Difficulty": difficulty,
            "Feature": feature,
            "n": len(group),
            "Mean_Normalized_Delta":
                delta.mean(),
            "Median_Normalized_Delta":
                delta.median(),
            "Positive_n":
                int((delta > 0).sum()),
            "Negative_n":
                int((delta < 0).sum())
        })

difficulty_normalized = pd.DataFrame(
    difficulty_rows
)

display(
    difficulty_normalized.round(6)
)


## 13. Redundancy among normalized paired features

In [ ]:

delta_norm_cols = [
    f"DeltaNorm_{feature}"
    for feature in FEATURES
]

normalized_delta_correlation = (
    paired_norm[
        delta_norm_cols
    ]
    .corr(
        method="spearman"
    )
)

display(
    normalized_delta_correlation
    .round(3)
)



## 14. Conservative candidate screen

This is an **analysis flag**, not a final feature-selection claim.

A feature is flagged as a robust candidate only when:

1. Holm-adjusted \(p<0.05\) after spline normalization;
2. Holm-adjusted \(p<0.05\) after linear normalization;
3. both normalizations give the same direction;
4. the normalized feature has only weak residual correlation with response length, \(|\rho|<0.20\);
5. its normalized paired difference also has only weak correlation with paired length difference, \(|\rho|<0.20\).

These criteria are deliberately conservative.


In [ ]:

candidate_screen = comparison.copy()

candidate_screen[
    "Weak_Residual_Length_Association"
] = (
    candidate_screen[
        "Normalized_rho_with_Words"
    ].abs() < 0.20
)

candidate_screen[
    "Weak_Delta_Length_Association"
] = (
    candidate_screen[
        "rho_DeltaNorm_vs_DeltaWords"
    ].abs() < 0.20
)

candidate_screen[
    "Robust_Candidate_Flag"
] = (
    candidate_screen[
        "Stable_Across_Normalizations"
    ]
    &
    candidate_screen[
        "Weak_Residual_Length_Association"
    ]
    &
    candidate_screen[
        "Weak_Delta_Length_Association"
    ]
)

display(
    candidate_screen[
        [
            "Feature",
            "Mean_Normalized_Delta",
            "Rank_Biserial",
            "Wilcoxon_p_Holm",
            "Mean_Normalized_Delta_Linear",
            "Rank_Biserial_Linear",
            "Wilcoxon_p_Linear_Holm",
            "Normalized_rho_with_Words",
            "rho_DeltaNorm_vs_DeltaWords",
            "Robust_Candidate_Flag"
        ]
    ]
    .sort_values(
        [
            "Robust_Candidate_Flag",
            "Wilcoxon_p_Holm"
        ],
        ascending=[
            False,
            True
        ]
    )
    .round(6)
)


## 15. Save Notebook 6B outputs

In [ ]:

OUTPUT_DIR = (
    "/content/Notebook6B_outputs"
)

os.makedirs(
    OUTPUT_DIR,
    exist_ok=True
)

norm_spline.to_csv(
    f"{OUTPUT_DIR}/"
    "Length_Normalized_Response_Features_Spline.csv",
    index=False
)

paired_norm.to_csv(
    f"{OUTPUT_DIR}/"
    "Length_Normalized_100_Paired_Differences.csv",
    index=False
)

length_diagnostics.to_csv(
    f"{OUTPUT_DIR}/"
    "Length_Normalization_Diagnostics.csv",
    index=False
)

normalized_stats.to_csv(
    f"{OUTPUT_DIR}/"
    "Length_Normalized_Paired_Tests.csv",
    index=False
)

delta_length_diagnostics.to_csv(
    f"{OUTPUT_DIR}/"
    "Normalized_Delta_Length_Diagnostics.csv",
    index=False
)

linear_sensitivity.to_csv(
    f"{OUTPUT_DIR}/"
    "Linear_Normalization_Sensitivity.csv",
    index=False
)

comparison.to_csv(
    f"{OUTPUT_DIR}/"
    "Spline_vs_Linear_Normalization.csv",
    index=False
)

difficulty_normalized.to_csv(
    f"{OUTPUT_DIR}/"
    "Normalized_Difficulty_Descriptives.csv",
    index=False
)

normalized_delta_correlation.to_csv(
    f"{OUTPUT_DIR}/"
    "Normalized_Delta_Feature_Correlation.csv"
)

candidate_screen.to_csv(
    f"{OUTPUT_DIR}/"
    "Robust_Candidate_Feature_Screen.csv",
    index=False
)

print(
    "Notebook 6B outputs saved."
)


In [ ]:

!zip -r /content/Notebook6B_Length_Normalization_Outputs.zip /content/Notebook6B_outputs

files.download(
    "/content/Notebook6B_Length_Normalization_Outputs.zip"
)



## Interpretation rule

Do **not** call a feature a hallucination marker merely because it was significant before normalization.

The strongest candidates for subsequent hypergraph/fuzzy modeling should be those that:
- remain significant after length normalization,
- show consistent direction under spline and linear normalization,
- have low residual dependence on length,
- and are not redundant with another stronger feature.

Notebook 7 should use these normalized results to decide which features enter the hypergraph/fuzzy stage.
